In [1]:
from IPython.display import clear_output
from Fetch_Live_Data import *
from Buy_Presure_Scaner import *
from Trade_Execution import *
from Chandelier_ZLSMA_Filter import *
from BTC_VIDYA import *
from EMAs_9_15_Filter import *
from Fetch_Coin_List import *

In [2]:
#coins = get_symbol()
#print("Total Coin Found: ", len(coins))

In [3]:
df = pd.read_csv("Coin_List.csv", header=None)
coins = df[1:].iloc[:, 0].tolist()

In [4]:
btc_trend = get_btc_volumatic_signal()
print(f"BTC Trend: {btc_trend}")

if btc_trend == "Green":
    chandelier_zlsma_out = chandelier_zlsma_filter_custom(
        coin_list=coins,
        timeframe='15m',
        max_buy_candles=3,
        zlsma_length=200
    )

    buy_presure = get_binance_buy_pressure(chandelier_zlsma_out['symbol'].tolist(), top_n=5)
    
    chandelier_zlsma_out = chandelier_zlsma_filter_custom(
        coin_list=buy_presure['symbol'].tolist(),
        timeframe='15m',
        max_buy_candles=3,
        zlsma_length=200
    )

BTC Trend: Green
Initializing Binance scanner...

Step 1: Scanning for Chandelier Exit buy signals...
Scanning 375 coins: ['PEPEUSDT', 'SHIBUSDT', 'BONKUSDT', 'XECUSDT', '1000SATSUSDT', 'WINUSDT', 'LUNCUSDT', 'FLOKIUSDT', 'DOGSUSDT', 'NEIROUSDT', 'LEVERUSDT', 'SPELLUSDT', 'DENTUSDT', 'HOTUSDT', 'HMSTRUSDT', '1MBABYDOGEUSDT', 'SLPUSDT', '1000CHEEMSUSDT', 'BOMEUSDT', 'MEMEUSDT', 'VTHOUSDT', 'NOTUSDT', 'MBLUSDT', 'SCUSDT', 'COSUSDT', 'IOSTUSDT', 'IQUSDT', 'TURBOUSDT', 'AMPUSDT', 'CKBUSDT', 'FUNUSDT', 'TLMUSDT', 'XVGUSDT', 'BEAMXUSDT', 'BANANAS31USDT', 'RSRUSDT', '1000CATUSDT', 'QKCUSDT', 'QIUSDT', 'CELRUSDT', 'DGBUSDT', 'PONDUSDT', 'REZUSDT', 'PENGUUSDT', 'ONEUSDT', 'ZILUSDT', 'GUSDT', 'DATAUSDT', 'JASMYUSDT', 'FIOUSDT', 'GALAUSDT', 'ANKRUSDT', 'REIUSDT', 'MDTUSDT', 'TUSDT', 'SUNUSDT', 'PEOPLEUSDT', 'ACHUSDT', 'SKLUSDT', 'RVNUSDT', 'ARPAUSDT', 'IDEXUSDT', 'RDNTUSDT', 'WAXPUSDT', 'IOTXUSDT', 'ALPHAUSDT', 'VETUSDT', 'GPSUSDT', 'BSWUSDT', 'BROCCOLI714USDT', 'ANIMEUSDT', 'ALTUSDT', 'ASTRUSDT'

In [5]:
chandelier_zlsma_out

,symbol,current_price,current_signal,buy_candles_count,signal_start_time,current_time,prev_sell_time,prev_sell_price,atr_value,long_stop,short_stop,zlsma_200,price_above_zlsma
0,IDUSDT,0.1701,BUY,3,2025-06-16 09:00:00,2025-06-16 09:30:00,2025-06-16 08:45:00,0.1685,0.001,0.1691,0.171,0.166173,0.003927
1,LAZIOUSDT,0.8280,BUY,3,2025-06-16 09:00:00,2025-06-16 09:30:00,2025-06-16 08:45:00,0.8250,0.006,0.8240,0.834,0.826828,0.003172
2,EPICUSDT,1.0470,BUY,3,2025-06-16 09:00:00,2025-06-16 09:30:00,2025-06-16 08:45:00,1.0450,0.004,1.0430,1.051,1.041602,0.005398


In [ ]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 500)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[480:]

def run_task():
    # Get the latest symbols every 2 hours
    symbols = pick_best_coin()
    return symbols

def fetch_and_display_data_(symbols):
    # Run the fetch_and_process_data every 10 seconds to display results
    result = fetch_and_process_data(symbols)
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    print(result)  # Display the new result
    print("\n")
    
    return result

# Main loop that runs every 2 hours
while True:
    symbols = run_task()  # Get the symbols every 2 hours (list of up to 10 symbols)
    print("New symbols received. Monitoring begins...\n")
    print(f"Symbols to monitor ({len(symbols)} symbols): {symbols}")
    
    trade_taken = False  # Flag to track if any trade was taken
    
    # Try each symbol one by one
    for i, symbol in enumerate(symbols):
        print(f"\n--- Checking symbol {i+1}/{len(symbols)}: {symbol} ---")
        
        # Continuous 10-second updates with fetched data for current symbol
        symbol_checked = False
        
        while not symbol_checked:
            out = fetch_and_display_data_([symbol])  # Pass single symbol as list
            
            # Check if the DataFrame is not empty and get the last row
            if not out.empty:
                # First condition: Check if buy_signal sum is <= 15
                if out['buy_signal'].sum() <= 15:
                    print(f"Buy signal sum condition met for {symbol}: {out['buy_signal'].sum()}")
                    
                    # Second condition: Check price above ZLSMA and buy signal
                    if out['close'].iloc[-1] > out['zlsma_200'].iloc[-1] and out['buy_signal'].iloc[-1] == True:
                        print(f"✅ All conditions met! Taking trade for: {symbol}")
                        #bot = SimpleATRTradingBot()
                        #result = bot.buy_signal(symbol, 10)
                        #status = bot.get_position_status()
                        print(f"✅ Trade Taken: {symbol}")
                        print("Running trade for 2 hours...")
                        trade_taken = True
                        symbol_checked = True  # Move to next phase
                        break  # Exit the symbol checking loop
                    else:
                        print(f"Buy signal sum ok but no buy signal detected for {symbol}")
                        symbol_checked = True  # Try next symbol
                else:
                    print(f"Buy signal sum > 15 for {symbol} (sum: {out['buy_signal'].sum()}), trying next symbol...")
                    symbol_checked = True  # Try next symbol
            else:
                print(f"The DataFrame is empty for {symbol}. No data available.")
                symbol_checked = True  # Try next symbol
        
        # If trade was taken, break out of symbol iteration
        if trade_taken:
            break
    
    # If no trade was taken with any symbol, refresh for new symbols
    if not trade_taken:
        print("\n❌ No trades taken with any symbols. Refreshing for new symbols...")
        continue  # Skip the 2-hour sleep and get new symbols immediately
    
    # If trade was taken, monitor for 2 hours with 10-second intervals
    print(f"\n🔄 Monitoring trade for 2 hours...")
    monitoring_start = time.time()
    
    while time.time() - monitoring_start < 2 * 60 * 60:  # 2 hours
        # You can add monitoring logic here if needed
        time.sleep(10)  # Wait for 10 seconds before next check
    
    print("2-hour monitoring period completed. Getting new symbols...")

In [ ]:

def pick_best_coin():
    scalping_filter = BinanceAllUSDTScalpingFilter(
        max_workers=8,
        delay_between_requests=0.1,
        weight_profile='volatile'  # Options: 'balanced', 'volatile', 'trending'
    )

    print("Scanning ALL Binance USDT pairs for scalping opportunities...")

    # get a list of dicts (or records)
    best_coins = scalping_filter.filter_all_usdt_pairs(
        min_volume=50_000,
        top_n=30,
        use_parallel=True,
        volume_filter_first=False,
        use_percentile_volume=True
    )

    # turn into a DataFrame
    df = pd.DataFrame(best_coins)
    df = df[df['current_price'] <= 50]

    # base filters: uptrend + cheap coins
    #base_mask = (df['trend_direction'] == 1) & (df['current_price'] <= 50)

    # nothing matched
    return df["symbol"].tolist()

In [ ]:
bot = SimpleATRTradingBot()
result = bot.buy_signal('BTC', 1000)
status = bot.get_position_status()